### Understanding magic for 1D XYZ Hamiltonian

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{j} (J_{x} \sigma_j^{x} \sigma_{j+1}^{x} + J_{y} \sigma_j^{y} \sigma_{j+1}^{y} + J_{z} \sigma_j^{z} \sigma_{j+1}^{z}) + \sum_{j} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

Magic often refers to the amount of non-clifford gates in the circuit. It has nothing to do with *Harry Potter*.

In [2]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import SuzukiTrotter, LieTrotter

from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
    generate_xyz_hamiltonian
)

import numpy as np
import scipy

In [3]:
num_qubits = 80 # increase the number to the largest linear chain you can get in the IBM Quantum Computer
Jx, Jy, Jz = np.pi/8, np.pi/4, np.pi/2 # feel free to change these parameters and see how the results change
h_x, h_y, h_z = np.pi/3, np.pi/6, np.pi/9 # feel free to change these parameters and see how the results change
dt = 0.01 # time evolution of each trotter step; feel free to increase or decrease and see how the results change
num_trotter_steps = 10 # decide on the number of trotter steps; do some trial and error to find the 2-qubit depth of the circuit for different trotter steps

Build a linear coupling map and construct the Hamiltonian from it

In [4]:
coupling_map = CouplingMap.from_line(num_qubits)

In [5]:
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(Jx, Jy, Jz),
    ext_magnetic_field=(h_x, h_y, h_z),
)
print(hamiltonian)

SparsePauliOp(['...'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j,
 0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j,
 0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j,
 0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j,
 0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.

Construct a half-filled initial state (also called Neel State). Note a Neel state looks like 010101...

In [6]:
init_state_neel = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    if i%2 == 0:
        init_state_neel.x(i)

Create Hamiltonian simulation circuit with Lie Trotter decomposition

In [7]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=dt*num_trotter_steps, # total time of evolution
    synthesis=LieTrotter(reps=num_trotter_steps), 
)

Compose this circuit with the initial state

In [8]:
circuit = init_state_neel.compose(circuit)

Let us select our observable as $O = \frac{1}{n}\sum_i Z_i$ acting on each qubit

In [9]:
from qiskit.quantum_info import SparsePauliOp
observable = SparsePauliOp(['I'*i + 'Z' + 'I'*(num_qubits-i-1) for i in range(num_qubits)], 
                            coeffs=[1/num_qubits]*num_qubits)
observable

SparsePauliOp(['ZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

#### Backpropagate the circuit using OBP

Here we are providing some guides to implement OBP. You should look into [this tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/operator-back-propagation) for more details on OBP.

In [10]:
from qiskit_addon_utils.slicing import slice_by_depth, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate

First we need to partition the circuit into slices. Backpropagation happens through each slice. Here we are using *slice_by_depth*. You are encouraged to try out other slicing mechanisms.

In [11]:
slices = slice_by_depth(circuit, max_slice_depth=1)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 91 slices.


Note that the original observable is $O = \frac{1}{n}\sum_i Z_i$ -- all the terms commute with each other. So we can simply run a single circuit, measure in $Z$ basis, and can calculate the expectation value of each term in post-processing. However, if our observable becomes, say $O = \frac{1}{n}\sum_i (Z_i + X_i)$, then, since $Z$ and $X$ don't commute with each other, we shall need two different circuits, one measured in $Z$ basis, and other measured in $X$ basis to calculate the two terms separately and then take their average. This implies that the quantum overhead increases with increasing number of non-commuting groups in the observable (since each non-commuting group will require its own circuit execution).

You can find the number of non-commuting groups of an observable simply by calling *.group_commuting(qubit_wise=True)* on the observable. For our observable $O$, we see that only one group is present.

In [12]:
observable.group_commuting(qubit_wise=True)

[SparsePauliOp(['ZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

When the operator backpropagates through the circuit, usually the number of non-commuting terms increases. Therefore, we need to set a maximum budget for the maximum number of circuits we are willing to execute.

In [13]:
op_budget = OperatorBudget(max_qwc_groups=10) # it says we are willing to execute at most 10 circuits
# feel free to change this number and see how the results change

In [14]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)

Backpropagated 6 slices.
New observable has 480 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 1024 terms across 12 groups.


**Q1**: Execute the original circuit and the backpropagated circuit and plot the expectation values

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
service = QiskitRuntimeService()
backend = service.least_busy(min_num_qubits=127)
backend

In [ ]:
from qiskit import generate_preset_pass_manager
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = # transpile the circuit
isa_obp_circuit = # transpile the aqc circuit
isa_observable = # apply the layout to the observable
isa_obp_observable = # apply the layout to the backpropagated observable

In [ ]:
estimator = EstimatorV2(mode=backend)
pubs = [(isa_circuit, isa_observable), (isa_obp_circuit, isa_obp_observable)]
job = estimator.run(pubs)

In [ ]:
result = job.result()[0]
obp_result = job.result()[1]

Obtain the expectation value and show them as a bar plot

**Q1**: Increase the max_budget to 20, 30 and 40 -- note the time required for backpropagating, the number of slices backpropagated, and the depth of the backpropagated circuit. From this, do a curve fitting, and provide an estimate of the max_budget required to backpropagate the whole circuit.

**Q2**: Execute each of the circuits from Q1, and show the expectation value and compare them against that of the original circuit.

**Q3**: At 80 qubits, the ideal expectation values are not known. One method to simulate quantum circuits is Pauli Propagation: https://github.com/Qiskit/pauli-prop. In this part, you will implement the same problem in Pauli Propagation; and compare the result as well as execution time with QPU. In particular, make use of [this](https://github.com/Qiskit/pauli-prop/blob/main/docs/tutorials/01_classically_estimate_expectation_values.ipynb) tutorial. Plot the ideal expectation value obtained from Pauli Propagation, and the computed expectation values from the original and the compressed circuits.

**Q5**: Next, we shall understand which direction of the external magnetic field makes the circuit more difficult to backpropagate. To understand this, we shall methodically remove one or more values of the $h$, and repeat the experiment. Let us start with removing $h_z$. This implies that the external magnetic field is aligned along $h_x + h_y$ plane.

***Why is this excercise useful?*** Did you connect the dot between OBP and Pauli Propagation? Yes, you are correct, Pauli Propagation is OBP where the backpropagation is through the entire circuit. And often such a method of classical simulation is way simpler than simulating the actual circuit. You cannot simulate a circuit beyond 32 qubits, but with Pauli Propagation we can simulate much larger cases. 

In Pauli propagation, the computation becomes difficult when the number of terms in the observable increase during backpropagation. And this happens when non-clifford gates (magic) are present. More the number of non-clifford gates, more is the difficulty to simulate the circuit through Pauli Propagation. Note that if you can easily backpropate -- which means this circuit can be easily simulated by Pauli Propagation, and has lesser chances of being a candidate for quantum advantage. This simple set of experiments allow you to decipher such a strong understanding.

In [ ]:
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(Jx, Jy, Jz),
    ext_magnetic_field=(h_x, h_y, 0), # h_z is set to 0
)
print(hamiltonian)

Repeat the above experiment and show how much you can backpropagate with max_qwc set to 10, 20, 30 and 40.

Now repeat the experiment by removing $h_x$ and $h_y$ values. This implies that the external magnetic field is along $h_z$ direction only.

**Q5**: *Using error mitigation*: The results from the hardware are noisy, and therefore may not be perfectly reliable. But we can use error mitigation to account for the noise and take the results from the original and the compressed circuit closer to the ideal. We shall use Probabilistic Error Amplification (PEA) and Twirled Readout Error Extinction (TREX) for this experiment. The first one accounts for Gate Errors, while the second one accounts for SPAM error.

For PEA, it is necessary to first learn the noise in the system. This can be done via NoiseLearner: https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/noise-learner-noise-learner. Also look into this tutorial: https://quantum.cloud.ibm.com/docs/en/tutorials/probabilistic-error-amplification#learn-the-noise-model-for-pea

In [ ]:
### Learn the noise in the system by running NoiseLearner

In [ ]:
### Run the circuit with PEA and TREX error mitigations and calculate the probability of occupancy for each qubit

**Q6**: *Using good noise factors for PEA*

A general issue with PEA is that if you use any random noise factor, the result may not improve. Therefore, it is important to understand which noise factors and extrapolators are good for the given circuit. A method to do that is to cliffordize the circuit, calculate the ideal result for the clifford circuit, try out different noise factors on the clifford circuit and select the one with the best result.

This can be performed using the NEAT tool: https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/debug-tools-neat

In [ ]:
#### Cliffordize the Hamiltonian simulation circuit

In [ ]:
### Calculate the ideal expectation value for the clifford circuit

In [ ]:
### Test for different noise factors and extrapolators
### Test with (1,3,5), (1,2,3), (1,1.2,1.4), (1,1.1,1.2) and for extrapolators linear, quadratic and exponential
### Find the noise factor and extrapolator that best matches the ideal expectation value for the clifford circuit

**NOTE** Since the depth of the original and the backpropagated circuits are different, the optimal noise factors may not be the same for the two. You need to run the above experiment separately for the two cases to find the best noise factors for each one of them.

In [ ]:
### Use the best noise factor and extrapolator to calculate the mitigated expectation values for the original and backpropagated cases
### Plot the ideal, noisy and mitigated expectation values for the original and backpropagated cases